In [ ]:
from sklearn.datasets import fetch_california_housing
import numpy as np 
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

def charger_immobilier():
    data = fetch_california_housing()
    print(f"California Housing : {data.data.shape} | features : {data.feature_names} | target médiane : {np.median(data.target)*10**5:.0f} $")
    return data.data, data.target

def evaluer_regression(modele, X_train, X_test, y_train, y_test):
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)
    return {
        "r2":   r2_score(y_test, y_pred),
        "mae":  mean_absolute_error(y_test, y_pred),
        "rmse": root_mean_squared_error(y_test, y_pred)
    }

# --- CAS NORMAL : dataset complet ---
print("\n=== CAS NORMAL ===")
X, y = charger_immobilier()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modele_lr = make_pipeline(StandardScaler(), LinearRegression())
modele_rf = make_pipeline(StandardScaler(), RandomForestRegressor(n_estimators=100, random_state=42))

resultats_lr = evaluer_regression(modele_lr, X_train, X_test, y_train, y_test)
resultats_rf = evaluer_regression(modele_rf, X_train, X_test, y_train, y_test)

print(f"LinearRegression : R2={resultats_lr['r2']:.2f}  MAE={resultats_lr['mae']:.2f}  RMSE={resultats_lr['rmse']:.2f}")
print(f"RandomForest     : R2={resultats_rf['r2']:.2f}  MAE={resultats_rf['mae']:.2f}  RMSE={resultats_rf['rmse']:.2f}")

# --- CAS LIMITE : 100 lignes aléatoires ---
print("\n=== CAS LIMITE : 100 lignes aléatoires ===")
np.random.seed(42)
idx = np.random.choice(len(X), 100, replace=False)
X100, y100 = X[idx], y[idx]
X_train100, X_test100, y_train100, y_test100 = train_test_split(X100, y100, test_size=0.2, random_state=42)

modele_lr2 = make_pipeline(StandardScaler(), LinearRegression())
modele_rf2 = make_pipeline(StandardScaler(), RandomForestRegressor(n_estimators=100, random_state=42))

r_lr2 = evaluer_regression(modele_lr2, X_train100, X_test100, y_train100, y_test100)
r_rf2 = evaluer_regression(modele_rf2, X_train100, X_test100, y_train100, y_test100)

print(f"LinearRegression : R2={r_lr2['r2']:.2f}  MAE={r_lr2['mae']:.2f}  RMSE={r_lr2['rmse']:.2f}")
print(f"RandomForest     : R2={r_rf2['r2']:.2f}  MAE={r_rf2['mae']:.2f}  RMSE={r_rf2['rmse']:.2f}")
print("→ R2 plus faible qu'avec le dataset complet pour le random forest : 100 lignes insuffisantes pour généraliser")
print("→ Le Random Forest mémorise les exemples d'entraînement sans généraliser")

# --- CAS ADVERSARIAL : quartier fictif ---
print("\n=== CAS ADVERSARIAL : quartier fictif ===")
# 8 variables : MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude
quartier_fictif = np.array([[0, 20, 5, 1, 9000, 3, 37.0, -122.0]])

prix_lr = modele_lr.predict(quartier_fictif)[0]
prix_rf = modele_rf.predict(quartier_fictif)[0]

print(f"LinearRegression prédit : {prix_lr*100_000:.0f} $")
print(f"RandomForest     prédit : {prix_rf*100_000:.0f} $")
print("→ Valeurs potentiellement absurdes : en production il faudrait valider les entrées (revenu >= 0, population réaliste)")


=== CAS NORMAL ===
California Housing : (20640, 8) | features : ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude'] | target médiane : 179700 $
LinearRegression : R2=0.58  MAE=0.53  RMSE=0.75
RandomForest     : R2=0.81  MAE=0.33  RMSE=0.51

=== CAS LIMITE : 100 lignes aléatoires ===
LinearRegression : R2=0.64  MAE=0.53  RMSE=0.64
RandomForest     : R2=0.65  MAE=0.47  RMSE=0.63
→ R2 plus faible qu'avec le dataset complet : 100 lignes insuffisantes pour généraliser
→ Le Random Forest mémorise les exemples d'entraînement sans généraliser

=== CAS ADVERSARIAL : quartier fictif ===
LinearRegression prédit : 68895 $
RandomForest     prédit : 182437 $
→ Valeurs potentiellement absurdes : en production il faudrait valider les entrées (revenu >= 0, population réaliste)
